# Validation

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
import torch
import pandas as pd
import os
import sys

from pathlib import Path

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    TASK_STATEMENTS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_LABELED_OUTPUT_PATH,
    TIMEZONES_LABELED_OUTPUT_PATH,
    TASK_MAPPING_LABELED_OUTPUT_PATH,
    LABOR_TRANSFER_LABELED_OUTPUT_PATH,
    JOB_ZONES_PATH,
    ExecutionMode,
    FINAL_LABELED_OUTPUT_PATH,
    SMALL_SAMPLE_INPUT_PATH,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
)
from validation import (
    work_related_metrics,
    agreement_rate,
    print_metrics,
)

In [4]:
# Device setup
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [5]:
# API client setup
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

In [6]:
execution_mode = ExecutionMode.DIRECT

## 2. Data Loading

In [7]:
# TODO: When we decide on the validation sample, swap out the
# sample_conversations_df with that

In [8]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [9]:
sample_df = sample_conversations(english_conversations)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (58777, 14)


In [10]:
sample_unique_df = preprocess_conversations(sample_df)
print(f"After dedup: {sample_unique_df.shape}")
sample_unique_df.head(2)

After dedup: (52489, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


In [11]:
labeled_work_related_df = pd.read_csv(SMALL_SAMPLE_INPUT_PATH)
labeled_work_related_df.shape

(24, 2)

In [12]:
labeled_work_related_df["conversation_str"] = labeled_work_related_df[
    "conversation"
].apply(str)
sample_unique_df["conversation_str"] = sample_unique_df["conversation"].apply(str)

In [13]:
sample_conversations_df = sample_unique_df.merge(
    labeled_work_related_df,
    on="conversation_str",
    how="right",
)
sample_conversations_df.shape

(24, 8)

In [14]:
sample_conversations_df = sample_conversations_df.drop(
    columns=["conversation_str", "conversation_y"]
)
sample_conversations_df = sample_conversations_df.rename(
    columns={"conversation_x": "conversation"}
)
sample_conversations_df.shape

(24, 6)

## 3. Work-Related Conversation Filtering

In [15]:
if not WORK_RELATED_LABELED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_LABELED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_LABELED_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_LABELED_OUTPUT_PATH)

sample_conversations_df["is_work_related_model"] = answers_df[
    "is_work_related_model"
].values

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

Work-related conversations: (15, 7)


,conversation,timestamp,country,state,hashed_ip,is_work_related_human,is_work_related_model
1,"[{'role': 'user', 'content': 'Context: making ...",2024-10-31 09:42:35,United States,Texas,0cd2c197eb582b7cacce9e802ad55615ccc3c7f10c78fb...,Yes,Yes
2,"[{'role': 'user', 'content': 'What metrics wou...",2024-03-07 01:18:38,United States,California,00a65413d886ad395f79b5d5ca4b96be87bd824ab2d72c...,Yes,Yes


In [36]:
sample_conversations_df["is_work_related_human"].value_counts()

is_work_related_human
Yes      13
No        8
Maybe     3
Name: count, dtype: int64

In [16]:
sample_conversations_df["is_work_related_model"].value_counts()

is_work_related_model
Yes      15
No        8
Maybe     1
Name: count, dtype: int64

In [17]:
metrics = work_related_metrics(
    y_true=sample_conversations_df["is_work_related_human"],
    y_pred=sample_conversations_df["is_work_related_model"],
)
print_metrics(metrics)

accuracy: 0.9167
fpr: 0.1818
tpr: 1.0000
precision: 0.8667
recall: 1.0000
cohen_kappa: 0.8471


In [18]:
disagreements_df = sample_conversations_df[
    sample_conversations_df["is_work_related_human"]
    != sample_conversations_df["is_work_related_model"]
].copy()
disagreements_df.to_csv("disagreements.csv", index=False)

## 4. Timezone conversion

In [19]:
work_related_df.head(5)

,conversation,timestamp,country,state,hashed_ip,is_work_related_human,is_work_related_model
1,"[{'role': 'user', 'content': 'Context: making ...",2024-10-31 09:42:35,United States,Texas,0cd2c197eb582b7cacce9e802ad55615ccc3c7f10c78fb...,Yes,Yes
2,"[{'role': 'user', 'content': 'What metrics wou...",2024-03-07 01:18:38,United States,California,00a65413d886ad395f79b5d5ca4b96be87bd824ab2d72c...,Yes,Yes
3,"[{'role': 'user', 'content': 'Write a LinkedIn...",2025-04-16 12:55:33,India,Maharashtra,afbfa4663345835405e32f91e94520a5ac9b9c40fbc643...,Yes,Yes
4,"[{'role': 'user', 'content': 'User: **Instruct...",2024-11-28 23:43:38,United States,NaN,17b9adaa3b447676b9618557679fc70eb56866205f59e5...,Yes,Yes
5,"[{'role': 'user', 'content': 'System: User: S...",2024-10-27 22:19:16,Mexico,Mexico City,5d7ced6eb67b18fa2af1791ac6f024aad747aec2200e83...,Yes,Yes


In [20]:
if not TIMEZONES_LABELED_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_LABELED_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_LABELED_OUTPUT_PATH)

if "timezone_y" in work_related_df.columns:
    work_related_df.drop(columns=["timezone_y"], inplace=True)
    work_related_df = work_related_df.rename(columns={"timezone_x": "timezone"})
work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

After timezone filter: (15, 8)


In [21]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

,timestamp,timezone,timestamp_local
0,2024-10-31 09:42:35+00:00,America/Chicago,2024-10-31 04:42:35-05:00
1,2024-03-07 01:18:38+00:00,America/Los_Angeles,2024-03-06 17:18:38-08:00
2,2025-04-16 12:55:33+00:00,Asia/Kolkata,2025-04-16 18:25:33+05:30


In [ ]:
work_related_df["conversation"].head(5)

0    [{'role': 'user', 'content': 'Context: making ...
1    [{'role': 'user', 'content': 'What metrics wou...
2    [{'role': 'user', 'content': 'Write a LinkedIn...
3    [{'role': 'user', 'content': 'User: **Instruct...
4    [{'role': 'user', 'content': 'System: \nUser: ...
Name: conversation, dtype: str

## 5. Task Mapping

In [23]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Management Occupations
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Management Occupations


In [24]:
if not TASK_MAPPING_LABELED_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_LABELED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_LABELED_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

Conversations to process: ['[{\'role\': \'user\', \'content\': \'Context: making a product page in XtreamTech.Net website! that sell IPTV subscriptions from differents IPTV Platforms.\\nTask: Write a compelling product description for an IPTV offer with title: Affordable Reliable mac portal code with VOD  ITALY IT TV channels,  using best SEO practices for 2024. Follow the structure outlined below and ensure the description is optimized for search engines to help it rank highly on Google. The output must be in the following JSON format:\\n{\\n  "excerpt": "A concise summary mentioning the main keywords of the post title: Affordable Reliable mac portal code with VOD  ITALY IT TV channels.",\\n  "introduction": "Introduction (1-2 sentences): Provide a brief introduction to the product, highlighting its main benefit and mentioning the keyword: Affordable Reliable mac portal code with VOD  ITALY IT TV channels.",\\n  "head1": "Shorten my title:\\\'Affordable Reliable mac portal code with V

Profession Mapping: 100%|██████████| 15/15 [01:47<00:00,  7.19s/it]


Running 15 direct Task mapping calls via OpenRouter...


Task Mapping: 100%|██████████| 15/15 [01:44<00:00,  6.95s/it]

Task mapped conversations: (15, 3)


,conversation,professions,tasks
0,"[{'role': 'user', 'content': 'Context: making ...","[Copy Writers, Search Marketing Strategists, M...",[Copy Writers:Write advertising copy for use b...
1,"[{'role': 'user', 'content': 'What metrics wou...","[Business Intelligence Analysts, Data Warehous...",[Business Intelligence Analysts:Synthesize cur...


In [26]:
task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[0] if pd.notnull(x) else None
)
task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[1] if pd.notnull(x) else None
)

print(f"After consensus filter: {task_mapped_df.shape}")
task_mapped_df.head(2)

After consensus filter: (8, 5)


,conversation,professions,tasks,job_title,selected_task
1,"[{'role': 'user', 'content': 'What metrics wou...","[Business Intelligence Analysts, Data Warehous...",Business Intelligence Analysts:Synthesize curr...,Business Intelligence Analysts,Synthesize current business intelligence or tr...
6,"[{'role': 'user', 'content': 'the following co...","[Bioinformatics Scientists, Bioinformatics Tec...",Bioinformatics Technicians:Enter or retrieve i...,Bioinformatics Technicians,Enter or retrieve information from structural ...


In [27]:
task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

task_mapped_df = task_mapped_df.merge(
    work_related_df.drop(columns=["conversation"]), on="conversation_str", how="inner"
)

task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final task mapped DataFrame: (8, 13)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_human,is_work_related_model,timezone,timestamp_local
0,"[{'role': 'user', 'content': 'What metrics wou...","[Business Intelligence Analysts, Data Warehous...",Business Intelligence Analysts:Synthesize curr...,Business Intelligence Analysts,Synthesize current business intelligence or tr...,2024-03-07 01:18:38+00:00,United States,California,00a65413d886ad395f79b5d5ca4b96be87bd824ab2d72c...,Yes,Yes,America/Los_Angeles,2024-03-06 17:18:38-08:00
1,"[{'role': 'user', 'content': 'the following co...","[Bioinformatics Scientists, Bioinformatics Tec...",Bioinformatics Technicians:Enter or retrieve i...,Bioinformatics Technicians,Enter or retrieve information from structural ...,2024-10-07 22:09:13+00:00,United States,Iowa,ce889ce3e16b9f11ee1fe84b4ac4d937da2ecdfe2e7747...,Yes,Yes,America/Chicago,2024-10-07 17:09:13-05:00


In [28]:
task_mapped_df.to_csv(TASK_MAPPING_LABELED_OUTPUT_PATH, index=False)

In [ ]:
# Execute this one only if human evaluation on task mapping has just been done and the file needs to be loaded for the
# next steps in the pipeline

# task_mapped_df = pd.read_csv(TASK_MAPPING_LABELED_OUTPUT_PATH)

In [32]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="selected_task_human_eval",
)
print(f"Human evaluators agreement with the task assignment: {agreement}")

Human evaluators agreement with the task assignment: 1.0


## 6. Labor Transfer Analysis

In [41]:
labor_transfer_df = pd.read_csv(TASK_MAPPING_LABELED_OUTPUT_PATH)
labor_transfer_df.shape

(8, 14)

In [42]:
if not LABOR_TRANSFER_LABELED_OUTPUT_PATH.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_LABELED_OUTPUT_PATH, index=False
    )
else:
    labor_transfer_labels = pd.read_csv(LABOR_TRANSFER_LABELED_OUTPUT_PATH)[
        "label"
    ].tolist()

print(f"Labor transfer labels: {len(labor_transfer_labels)}")
labor_transfer_df["labor_transfer"] = labor_transfer_labels
print(f"Labor transfer labels assigned: {labor_transfer_df.shape}")

Running 8 direct Labor Transfer calls via OpenRouter...


Labor Transfer: 100%|██████████| 8/8 [00:28<00:00,  3.58s/it]

Labor transfer labels: 8
Labor transfer labels assigned: (8, 15)


In [43]:
labor_transfer_df = expand_labor_transfer_labels(
    df=labor_transfer_df, label_column="labor_transfer"
)
print(f"Final DataFrame: {labor_transfer_df.shape}")
labor_transfer_df.head(2)

Final DataFrame: (8, 22)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_human,...,timestamp_local,selected_task_human_eval,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence
0,"[{'role': 'user', 'content': 'What metrics wou...","['Business Intelligence Analysts', 'Data Wareh...",Business Intelligence Analysts:Synthesize curr...,Business Intelligence Analysts,Synthesize current business intelligence or tr...,2024-03-07 01:18:38+00:00,United States,California,00a65413d886ad395f79b5d5ca4b96be87bd824ab2d72c...,Yes,...,2024-03-06 17:18:38-08:00,Yes,consumer,good,LT1,unclear_user_role,other,business_intelligence_analyst,"The user requested BI metrics, data sources, a...",medium
1,"[{'role': 'user', 'content': 'the following co...","['Bioinformatics Scientists', 'Bioinformatics ...",Bioinformatics Technicians:Enter or retrieve i...,Bioinformatics Technicians,Enter or retrieve information from structural ...,2024-10-07 22:09:13+00:00,United States,Iowa,ce889ce3e16b9f11ee1fe84b4ac4d937da2ecdfe2e7747...,Yes,...,2024-10-07 17:09:13-05:00,Yes,consumer,good,LT2,NaN,other,bioinformatics technician,The user provided bacterial genome metadata an...,high


In [44]:
labor_transfer_df.to_csv(LABOR_TRANSFER_LABELED_OUTPUT_PATH, index=False)

In [ ]:
# Execute this one only if the labor transfer analysis has just been done and the file needs to be re-loaded for the next
# steps in the pipeline

# labor_transfer_df = pd.read_csv(LABOR_TRANSFER_LABELED_OUTPUT_PATH)

In [47]:
agreement = agreement_rate(
    df=labor_transfer_df,
    column="labor_transfer_human_eval",
)
print(agreement)

0.625


# 7. Job Zones

In [25]:
job_zones_df = pd.read_excel(JOB_ZONES_PATH)
job_zones_df.head(2)

,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst


In [26]:
task_mapped_df = task_mapped_df.merge(
    job_zones_df[["Title", "Job Zone"]],
    left_on="job_title",
    right_on="Title",
    how="left",
)
task_mapped_df = task_mapped_df.drop(columns=["Title"])
print(f"After merging job zones: {task_mapped_df.shape}")
task_mapped_df.head(2)

After merging job zones: (9, 21)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,...,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence,Job Zone
0,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,...,2024-11-04 05:44:03+04:30,consumer,good,LT1,low_stakes,translator,NaN,User requested translation of a short figure c...,high,4.0
1,"[{'role': 'user', 'content': 'You are a helpfu...","[Instructional Designers and Technologists, Te...",Instructional Designers and Technologists:Deve...,Instructional Designers and Technologists,Develop instructional materials and products f...,2024-11-09 09:44:50+00:00,Belgium,Brussels Capital,5a527b1cb1af88eb990a824042cb105bb336ee03606f4f...,Yes,...,2024-11-09 10:44:50+01:00,consumer,good,LT1,unclear_user_role,other,instructional designer,User asks ChatGPT to generate a synthetic data...,medium,NaN


In [27]:
final_df = task_mapped_df.copy()
final_df.to_csv(FINAL_LABELED_OUTPUT_PATH, index=False)